# superGroups · rehearsal room
Train virtual musicians, build a supergroup, and compare how its members play together.

**MIDI research prototype:** the included musicians learn synthetic patterns. The WAV player uses a simple preview synth, not learned instrument timbres. To learn your own musicians, supply labelled multitrack MIDI.

Choose **Runtime → Change runtime type → T4 GPU**. Run the cells in order. Training uses your allocated Colab compute. The default run is 1,000 optimizer updates; no runtime estimate is promised. Checkpoints are written every 100 updates. Set `USE_DRIVE=True` to preserve them across runtime deletion.


In [ ]:
from pathlib import Path
import subprocess, sys, os
REPO = Path('/content/nicjams.github.io')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/nicjams/nicjams.github.io.git', str(REPO)], check=True)
PROJECT = REPO / 'superGroups'
os.chdir(PROJECT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', 'matplotlib', 'ipywidgets'], check=True)
import torch
print('PyTorch:', torch.__version__)
assert torch.cuda.is_available(), 'Select a GPU runtime before training, then rerun this cell.'
print('GPU:', torch.cuda.get_device_name(0))
print('GPU memory (GiB):', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
def run(*args):
    subprocess.run([sys.executable, '-m', 'supergroups.cli', *map(str, args)], check=True)


## Storage and data
The demo makes 256 original procedural performances across 12 virtual musicians. For custom data, set `DATA_MODE='custom'` and supply a manifest path. See the project README for the track labelling format. A musician needs performances in training; evaluation holds out whole songs.


In [ ]:
USE_DRIVE = False  # Change to True for persistent storage; Colab will ask you to mount Drive.
DATA_MODE = 'demo'  # 'demo' or 'custom'
MANIFEST = '/content/drive/MyDrive/superGroups/manifest.json'
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/superGroups') if USE_DRIVE else Path('/content/superGroups-session')
ROOT.mkdir(parents=True, exist_ok=True)
DATA = ROOT / f'{DATA_MODE}.npz'
RUN = ROOT / f'{DATA_MODE}-run'
if DATA_MODE == 'demo':
    run('demo-data', '--out', DATA, '--songs', 256, '--steps', 128)
elif DATA_MODE == 'custom':
    run('prepare', MANIFEST, '--out', DATA, '--steps', 128)
else:
    raise ValueError('Choose demo or custom')
print('Training data:', DATA)
print('Checkpoints:', RUN)


## Train
FP16 mixed precision, gradient accumulation, clipping, grouped validation, and resumable checkpoints are built in. `RESUME=True` restores `last.pt`; `UPDATES` is the number of **additional** updates. Keep the same data and split seed. If memory runs out, lower the batch size to 1. Do not interpret falling loss alone as good music: inspect onset F1 and listen to samples.


In [ ]:
UPDATES = 1000
BATCH_SIZE = 4
RESUME = False
command = ['train', '--data', DATA, '--out', RUN, '--updates', UPDATES,
           '--batch-size', BATCH_SIZE, '--accumulate', 4, '--eval-every', 100, '--device', 'cuda']
if RESUME:
    command += ['--resume', RUN / 'last.pt']
run(*command)


In [ ]:
import json
import matplotlib.pyplot as plt
metrics = [json.loads(line) for line in (RUN / 'metrics.jsonl').read_text().splitlines()]
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
for name in ['train_loss', 'val_loss']:
    axes[0].plot([m['update'] for m in metrics], [m[name] for m in metrics], label=name)
axes[0].legend(); axes[0].set_title('Weighted categorical loss'); axes[0].set_xlabel('Optimizer updates')
axes[1].plot([m['update'] for m in metrics], [m['onset_f1'] for m in metrics])
axes[1].set_title('Held-out onset F1 (teacher forced)'); axes[1].set_ylim(0, 1); axes[1].set_xlabel('Optimizer updates')
plt.show()


## Assemble the band
Pick one trained identity per instrument. Generate a rehearsal, then change one member and regenerate using the same seed to compare. Each musician hears the others through the current time step and their own previous notes. Two rehearsal rounds let everyone respond to the latest arrangement.


In [ ]:
import ipywidgets as widgets
from IPython.display import display, Audio
from supergroups.train import load_checkpoint
from supergroups.model import ROLES
from supergroups.generate import perform
from supergroups.data import write_midi
from supergroups.audio import render
model, checkpoint = load_checkpoint(RUN / 'best.pt', 'cuda')
selectors = [widgets.Dropdown(options=[m['name'] for i, m in enumerate(checkpoint['registry'])
              if m['role'] == role and i in checkpoint['trained_ids']], description=role.title()) for role in ROLES]
defaults = ['pocket-bass', 'spacious-keys', 'restless-lead', 'pocket-drums']
for selector, default in zip(selectors, defaults):
    if default in selector.options: selector.value = default
seed = widgets.IntText(value=17, description='Seed')
temperature = widgets.FloatSlider(value=.85, min=.2, max=1.5, step=.05, description='Temperature')
rounds = widgets.IntSlider(value=2, min=1, max=4, description='Rehearsals')
display(widgets.VBox(selectors + [seed, temperature, rounds]))


In [ ]:
import numpy as np
names = [s.value for s in selectors]
lookup = {m['name']: i for i, m in enumerate(checkpoint['registry'])}
identities = [lookup[name] for name in names]
roll = perform(model, identities, model.config.steps, rounds.value, temperature.value, seed.value)
write_midi(roll, RUN / 'supergroup.mid', names=names)
render(RUN / 'supergroup.mid', RUN / 'supergroup.wav')
(RUN / 'supergroup.json').write_text(json.dumps({'lineup': names, 'seed': seed.value,
    'temperature': temperature.value, 'rounds': rounds.value, 'training_source': checkpoint['source']}, indent=2))
print(' + '.join(names))
display(Audio(filename=str(RUN / 'supergroup.wav')))
fig, axes = plt.subplots(4, 1, figsize=(14, 7), sharex=True)
for role, ax in enumerate(axes):
    ax.imshow((roll[:, role] > 0).T, origin='lower', aspect='auto', interpolation='nearest', cmap='magma')
    ax.set_ylabel(names[role]); ax.set_ylim(24, 100)
axes[-1].set_xlabel('Sixteenth-note steps')
plt.tight_layout(); plt.show()


## Interaction experiment
The next cell holds the lineup, seed, temperature, and number of rounds fixed, but removes the other musicians from each player's listening context. Compare the result to the previous performance. Differences measure sensitivity, not necessarily musical improvement. In another experiment, swap only the keys player above and rerun the previous cell.


In [ ]:
independent = perform(model, identities, model.config.steps, rounds.value, temperature.value, seed.value, listening=False)
write_midi(independent, RUN / 'independent.mid', names=names)
render(RUN / 'independent.mid', RUN / 'independent.wav')
print('Playing without hearing bandmates:')
display(Audio(filename=str(RUN / 'independent.wav')))
for role, name in enumerate(names):
    print(name, '| onsets listening:', int((roll[:, role] >= 2).sum()),
          '| independent:', int((independent[:, role] >= 2).sum()),
          '| changed pitch/steps:', int((roll[:, role] != independent[:, role]).sum()))


## Keep the results
This archive contains the model, optimizer state, metrics, generated MIDI, and previews. If you used temporary runtime storage, download it before disconnecting. The training dataset is separate; retain it too if you want reproducible resume. Use MIDI with a DAW or SoundFont for higher-quality sound.


In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive(str(ROOT / 'superGroups-results'), 'zip', RUN)
files.download(archive)
